# COMP5329 — Week 10 Self-Study

# Deep Reinforcement Learning: From MDPs to DeepSeek-R1

**Semester 1, 2026**

---

## How to use this material

This notebook is a **standalone, comprehensive companion** to the Week 10 lecture. It is not a supplement to the 60-minute in-class tutorial — it is an independent document that covers **every concept on the lecture slides** in depth, with mathematical derivations and (where useful) short PyTorch snippets that illustrate the engineering view.

| Document | Purpose | Time budget |
|---|---|---|
| `Week 10-Deep Reinforcement Learning.pptx` | Lecture slides (ground truth for content scope) | 90 min lecture |
| `Week10_Deep_Reinforcement_Learning.ipynb` | In-class tutorial (tutor review + practice + exam questions) | 60 min tutorial |
| **`Week10_Self_Study_Deep_RL.ipynb`** (this file) | **Full self-study** — covers everything on the slides, even if also discussed in the tutorial | Self-paced, 3–5 hours |

If a topic appears in *both* the tutorial and here, that is **on purpose**: the tutorial is optimised for a 60-minute live session, while this document is optimised for deep understanding at your own pace. When the two overlap, this document goes deeper.

## Prerequisites

- **Week 6** — MLPs, backprop, SGD (you will derive TD-learning gradients).
- **Week 7** — Transformers (you will reason about them as autoregressive policies over tokens).
- **Week 9** — Alignment overview (pre-training → SFT → RLHF / DPO). This notebook fills in the RL machinery that Week 9 treated as a black box.

## Roadmap

**Part I — Classical RL.** Chapters 1–7 build RL from scratch: history, MDPs, returns and discounting, value functions and the Bellman equations, tabular Q-learning, and Deep Q-Networks. No LLMs yet.

**Part II — RL in the age of LLMs.** Chapters 8–17 port every classical concept to language models: the token-level MDP, reward models via Bradley–Terry, RLHF's full PPO machinery, and then three simplifications — DPO, SLiC-HF, and GRPO — culminating in the DeepSeek-R1 case study.

Chapter 18 lists references and recommended further reading.


In [ ]:
# ── Imports used throughout this notebook ──────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)


---

# Part I — Classical Reinforcement Learning

Chapters 1–7 cover classical RL: the mathematical framework (MDP, returns, Bellman), the first tabular algorithm (Q-learning), and its deep extension (DQN). These seven chapters correspond to slides 1–22 of the lecture.


## Chapter 1 — Motivation and Historical Breakthroughs

*(Lecture slides 1–3)*

### 1.1 Why reinforcement learning?

Supervised learning assumes you have a dataset of $(x, y)$ pairs: someone has already annotated the right answer for every input. Many problems are not like that:

- **Games.** Nobody labels "the best move" for every board position in Go — the only ground truth is "did you win or lose?", which arrives many moves later.
- **Robotics.** A robot grasping an object doesn't get a per-joint-angle label. It gets a success/failure signal at the end of the attempt.
- **Dialogue and language.** There is no "correct answer" to the prompt *"Write a poem about autumn"*. There are only degrees of human preference.

Reinforcement learning is the mathematical framework for these problems: an **agent** interacts with an **environment**, takes **actions**, and receives a scalar **reward** signal that it must learn to maximise over time. No labels, just consequences.

### 1.2 Three decades of breakthroughs

| Year | System | Domain | Key idea |
|---|---|---|---|
| 1992 | TD-Gammon (Tesauro) | Backgammon | Neural network + TD-learning, reaches expert level via self-play |
| 2013–15 | DQN (DeepMind) | 49 Atari games | CNN Q-network from raw pixels + experience replay |
| 2016 | AlphaGo | Go | Policy/value CNNs + MCTS, beats Lee Sedol 4–1 |
| 2017 | AlphaGo Zero | Go | Learns from zero human data via self-play |
| 2018 | AlphaZero | Chess, shogi, Go | One architecture, three games, all superhuman |
| 2018 | OpenAI Five | Dota 2 | Multi-agent RL + PPO at massive scale |
| 2019 | AlphaStar | StarCraft II | Real-time RTS with imperfect information |
| 2020–22 | InstructGPT / ChatGPT | Language | RLHF — RL for aligning LLMs to human preferences |
| 2025 | DeepSeek-R1 | Reasoning LLMs | GRPO + rule-based rewards → emergent chain-of-thought |

The trajectory is clear: RL started with board games where the reward was unambiguous (win/loss), then expanded to video games where the state space became enormous (pixels), and finally arrived at language — where the reward itself has to be learned from humans. Chapters 2–7 cover the first half of this trajectory; Part II covers the second.

### 1.3 RL in a nutshell

```
            ┌─────────────┐
            │   Agent     │   ← policy π(a | s)
            └─────────────┘
               ▲       │
       reward  │       │ action
          r_t  │       │ a_t
               │       ▼
            ┌─────────────┐
            │ Environment │
            └─────────────┘
               │
      new state s_{t+1}
```

At each discrete time step $t$:
1. The agent observes state $s_t$.
2. The agent picks action $a_t$ according to its policy $\pi(a_t \mid s_t)$.
3. The environment returns reward $r_{t+1}$ and a new state $s_{t+1}$.
4. Repeat until the episode ends (or forever).

The goal is to choose $\pi$ so that the sum of future rewards is as large as possible. The next four chapters turn this picture into precise mathematics.


## Chapter 2 — Markov Decision Processes

*(Lecture slides 4–5)*

### 2.1 Formal definition

A **Markov Decision Process (MDP)** is a 5-tuple $(S, A, P, R, \gamma)$:

| Symbol | Name | Meaning |
|---|---|---|
| $S$ | State space | Set of possible states. Can be discrete (grid world) or continuous (robot joint angles). |
| $A$ | Action space | Set of possible actions. Can also be discrete or continuous. |
| $P(s' \mid s, a)$ | Transition kernel | Probability of landing in $s'$ after taking $a$ in $s$. |
| $R(s, a, s')$ | Reward function | Immediate scalar reward for the transition. Sometimes written $R(s, a)$ or $R(s)$. |
| $\gamma \in [0, 1)$ | Discount factor | How much to value future reward relative to immediate reward (Chapter 3). |

### 2.2 The Markov property

The defining assumption of an MDP is:
$$P(s_{t+1} \mid s_t, a_t, s_{t-1}, a_{t-1}, \ldots, s_0, a_0) = P(s_{t+1} \mid s_t, a_t).$$

In words: **the current state is a sufficient statistic for the future.** Once you know $s_t$, history gives you no additional information about what happens next.

This is stronger than it sounds. "The last four frames of an Atari screen" is Markovian; "the last one frame" is *not* (you cannot tell which direction a ball is moving from a single frame). The engineering art of applying RL is often the art of constructing Markovian state representations.

### 2.3 Policies

A **policy** is a rule for acting. There are two flavours:

- **Deterministic policy:** $a = \mu(s)$. One action per state.
- **Stochastic policy:** $\pi(a \mid s)$ — a probability distribution over actions.

Stochastic policies are strictly more general and are necessary when:
1. The optimal policy is itself stochastic (e.g., rock–paper–scissors).
2. We need exploration during learning (ε-greedy, Chapter 5).
3. We want to back-propagate through the policy (policy gradients, Part II).

### 2.4 A tiny MDP you can hold in your head

Let's make all of this concrete with a 3-state MDP. States = $\{s_0, s_1, s_2\}$, actions = $\{L, R\}$. Transitions are deterministic; reward is $+1$ on arriving at $s_2$ (terminal), $0$ elsewhere; $\gamma = 0.9$.


In [ ]:
# ── A tiny 3-state MDP ──────────────────────────────────────────────────────

class TinyMDP:
    """States 0, 1, 2 (terminal). Actions: 0 = L, 1 = R. Reward +1 on reaching 2."""
    n_states = 3
    n_actions = 2
    terminal = 2

    @staticmethod
    def step(s, a):
        # deterministic transitions
        if s == 0:
            s_next = 0 if a == 0 else 1       # L: stay;  R: go right
        elif s == 1:
            s_next = 0 if a == 0 else 2       # L: back; R: reach terminal
        else:
            s_next = 2                         # terminal absorbs
        r = 1.0 if (s_next == TinyMDP.terminal and s != TinyMDP.terminal) else 0.0
        done = (s_next == TinyMDP.terminal)
        return s_next, r, done


# Roll out one episode with a random policy
env = TinyMDP()
s, steps = 0, []
while True:
    a = np.random.randint(2)
    s_next, r, done = env.step(s, a)
    steps.append((s, a, r, s_next))
    s = s_next
    if done or len(steps) > 20:
        break
for t, (s, a, r, s_next) in enumerate(steps):
    act_name = 'L' if a == 0 else 'R'
    print(f't={t}  s={s}  a={act_name}  r={r}  s_next={s_next}')


## Chapter 3 — Returns and Discounting

*(Lecture slide 6)*

The **return** $G_t$ is the total reward accumulated from time step $t$ onwards. The agent's job is to maximise the expected return. But there is a subtle problem — "total reward" can mean different things depending on whether tasks are episodic or continuing, and whether we discount.

### 3.1 Episodic vs continuing tasks

- **Episodic tasks** have a well-defined terminal state (Go game ends, robot drops the object, episode reaches time limit). The return is a finite sum:
  $$G_t = r_{t+1} + r_{t+2} + \ldots + r_T.$$
- **Continuing tasks** have no natural end (load balancer running forever, lifelong robot). The natural sum is infinite:
  $$G_t = \sum_{k=0}^{\infty} r_{t+k+1}.$$

### 3.2 Why discount?

The infinite sum above can diverge. If every step gives reward $r_{\max} > 0$, then $G_t = \infty$ for every policy — the gradient of "total reward" with respect to anything is undefined, and comparing policies becomes meaningless.

The fix is the **discounted return**:
$$G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1}, \qquad \gamma \in [0, 1).$$

The discount factor $\gamma < 1$ buys us three things at once:

**(a) Mathematical convergence.** If rewards are bounded by $r_{\max}$, then
$$|G_t| \leq \sum_{k=0}^{\infty} \gamma^k r_{\max} = \frac{r_{\max}}{1 - \gamma}.$$
The return is finite and the theory of value functions, contraction mappings, and Q-learning convergence all go through.

**(b) Preference for near-term reward.** A reward $r$ that arrives $k$ steps from now contributes $\gamma^k r$. So $\gamma$ encodes "how patient" the agent is. A self-driving car with $\gamma = 0.99$ thinks about rewards ~100 steps ahead; a short-sighted day-trading bot with $\gamma = 0.5$ thinks two steps ahead.

**(c) Implicit horizon.** Even in episodic tasks where convergence is not a concern, $\gamma < 1$ acts like a "soft termination": at step $k$, the effective probability of surviving is $\gamma^k$. This simplifies analysis and is how many algorithms (including Q-learning) actually work.

### 3.3 A numerical look at discounting


In [ ]:
# ── How γ reshapes a trajectory's return ────────────────────────────────────

rewards = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 10.0])   # delayed reward of +10 at t=9

for gamma in [0.0, 0.5, 0.9, 0.99, 1.0]:
    discounts = gamma ** np.arange(len(rewards))
    G = (discounts * rewards).sum()
    print(f'gamma = {gamma:4.2f}   G_0 = {G:7.4f}    (final reward weighted by γ^9 = {gamma**9:.4g})')


Observe:

- With **$\gamma = 0$** the agent is myopic — a reward 9 steps away counts as zero. It cannot learn about any consequence that isn't instantaneous.
- With **$\gamma = 1$** the return is just the raw sum. Fine for this short episode, but would diverge for continuing tasks.
- Between those extremes, $\gamma$ smoothly controls how far into the future the agent looks. Typical values in practice are $\gamma \in [0.9, 0.999]$.

In the tutorial's grid world we used $\gamma = 0.99$ — this was not an arbitrary choice; it allowed the $+10$ goal reward to propagate backwards through all 25 cells with a meaningful gradient.


## Chapter 4 — Value Functions and the Bellman Equations

*(Lecture slides 7–9)*

### 4.1 Two kinds of value function

Given a policy $\pi$, there are two natural prediction targets:

**State-value function** — "How much reward will I get *from state $s$* if I follow $\pi$?"
$$V^\pi(s) = \mathbb{E}_\pi\!\left[\sum_{k=0}^{\infty} \gamma^k r_{t+k+1} \;\middle|\; s_t = s\right].$$

**Action-value function (Q-function)** — "How much reward will I get from taking action $a$ in state $s$, and then following $\pi$?"
$$Q^\pi(s, a) = \mathbb{E}_\pi\!\left[\sum_{k=0}^{\infty} \gamma^k r_{t+k+1} \;\middle|\; s_t = s, a_t = a\right].$$

The two are related:
$$V^\pi(s) = \sum_a \pi(a \mid s)\, Q^\pi(s, a).$$

Q-functions are more useful for *acting*: given $Q^\pi$, the best action in state $s$ is simply $\arg\max_a Q^\pi(s, a)$ — no model of the environment required. This is why tabular RL algorithms learn $Q$, not $V$.

### 4.2 Deriving the Bellman expectation equation

Start from the definition of $V^\pi$ and peel off the first reward:

$$
\begin{aligned}
V^\pi(s) &= \mathbb{E}_\pi\!\left[r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \ldots \;\middle|\; s_t = s\right] \\
&= \mathbb{E}_\pi\!\left[r_{t+1} + \gamma \underbrace{(r_{t+2} + \gamma r_{t+3} + \ldots)}_{=\,G_{t+1}} \;\middle|\; s_t = s\right] \\
&= \mathbb{E}_\pi\!\left[r_{t+1} + \gamma V^\pi(s_{t+1}) \;\middle|\; s_t = s\right].
\end{aligned}
$$

Taking the expectation over $\pi$ and $P$ explicitly:

$$\boxed{\;V^\pi(s) = \sum_a \pi(a \mid s) \sum_{s'} P(s' \mid s, a)\bigl[R(s, a, s') + \gamma V^\pi(s')\bigr]\;}$$

This is the **Bellman expectation equation**. It says: *the value of a state equals the immediate reward plus the discounted value of the next state* — averaged over the policy and the environment.

The analogous equation for $Q^\pi$ is obtained by not averaging over $a$:

$$Q^\pi(s, a) = \sum_{s'} P(s' \mid s, a)\!\left[R(s, a, s') + \gamma \sum_{a'} \pi(a' \mid s') Q^\pi(s', a')\right].$$

### 4.3 The Bellman optimality equation

Among all policies, there is always one (possibly non-unique) **optimal policy** $\pi^*$ whose value function dominates every other policy's value function at every state. We write $V^*(s) = V^{\pi^*}(s)$ and $Q^*(s, a) = Q^{\pi^*}(s, a)$.

The optimal Q-function satisfies a *stronger* recursive equation, the **Bellman optimality equation**:

$$\boxed{\;Q^*(s, a) = \sum_{s'} P(s' \mid s, a)\!\left[R(s, a, s') + \gamma \max_{a'} Q^*(s', a')\right]\;}$$

The difference is the $\max$: instead of averaging over $\pi$, we take the *best* action at the next step. And once we have $Q^*$, the optimal policy is trivial:
$$\pi^*(s) = \arg\max_a Q^*(s, a).$$

### 4.4 Why this matters: Bellman is a contraction

Define the **Bellman optimality operator** $\mathcal{T}$ acting on Q-functions:
$$(\mathcal{T} Q)(s, a) = \sum_{s'} P(s' \mid s, a)\!\left[R(s, a, s') + \gamma \max_{a'} Q(s', a')\right].$$

One can show (Banach fixed-point theorem) that $\mathcal{T}$ is a $\gamma$-contraction in the sup norm: for any two Q-functions,
$$\|\mathcal{T} Q_1 - \mathcal{T} Q_2\|_\infty \leq \gamma \|Q_1 - Q_2\|_\infty.$$

Two consequences follow immediately:

1. **$Q^*$ is the unique fixed point** of $\mathcal{T}$: $\mathcal{T} Q^* = Q^*$.
2. **Iterating $\mathcal{T}$ converges from any start:** the sequence $Q_{k+1} = \mathcal{T} Q_k$ has $\|Q_k - Q^*\|_\infty \leq \gamma^k \|Q_0 - Q^*\|_\infty \to 0$.

This is the theoretical basis of Q-learning, DQN, and indeed every value-based RL algorithm. When they update Q towards a bootstrapped target, they are *approximating one application of $\mathcal{T}$*.

### 4.5 Value iteration on the tiny MDP from Chapter 2


In [ ]:
# ── Value iteration on the 3-state MDP ──────────────────────────────────────

gamma = 0.9
Q = np.zeros((3, 2))  # states × actions
env = TinyMDP()

for it in range(40):
    Q_new = np.zeros_like(Q)
    for s in range(3):
        if s == TinyMDP.terminal:
            continue
        for a in range(2):
            s_next, r, done = env.step(s, a)
            bootstrap = 0.0 if done else np.max(Q[s_next])
            Q_new[s, a] = r + gamma * bootstrap
    delta = np.abs(Q_new - Q).max()
    Q = Q_new
    if delta < 1e-9:
        print(f'Converged in {it + 1} iterations'); break

print('Q*(s, a):')
print('         L         R')
for s in range(3):
    print(f's={s}:  {Q[s,0]:8.4f}  {Q[s,1]:8.4f}')

print()
print('Optimal actions at non-terminal states:',
      ['L' if np.argmax(Q[s]) == 0 else 'R' for s in range(3) if s != TinyMDP.terminal])


A few observations worth internalising:

- $Q^*(0, R) = \gamma \cdot Q^*(1, R) = \gamma \cdot 1 = 0.9$. The reward $+1$ gets discounted by one step as it propagates back from $s_1$ to $s_0$.
- $Q^*(0, L) = \gamma \cdot Q^*(0, R) = \gamma^2 = 0.81$ — going left stays at $s_0$, so the reward is delayed by one more step.
- The policy $\pi^*(s) = R$ for both non-terminal states is optimal, and we never needed a "policy network" to find it — value iteration on $Q$ gave us everything.

### 4.6 From value iteration to Q-learning

Value iteration requires knowing $P$ and $R$ — the full model of the environment. In most interesting problems we don't have them. Q-learning (Chapter 5) is the **sampling-based** version of value iteration: it replaces the expectation $\sum_{s'} P(s' \mid s, a)[\ldots]$ with a single observed transition $(s, a, r, s')$, and uses a running average to converge.


## Chapter 5 — Tabular Q-Learning

*(Lecture slides 10–14)*

### 5.1 The Q-table

When $|S|$ and $|A|$ are both small and discrete, we can store $Q(s, a)$ in a literal table — one row per state, one column per action:

| $s$ \\ $a$ | $a_1$ | $a_2$ | $a_3$ | $\ldots$ |
|---|---|---|---|---|
| $s_1$ | $Q_{11}$ | $Q_{12}$ | $Q_{13}$ | $\ldots$ |
| $s_2$ | $Q_{21}$ | $Q_{22}$ | $Q_{23}$ | $\ldots$ |
| $\vdots$ |  |  |  | $\ddots$ |

Updating the table means writing into one cell; "inference" means reading a row and picking its max. Everything is $O(|A|)$.

### 5.2 The Q-learning update rule

Q-learning turns the Bellman optimality equation into a learning rule. Given an observed transition $(s, a, r, s')$, it performs a **partial step** of value iteration restricted to that one cell:

$$Q(s, a) \leftarrow Q(s, a) + \alpha\underbrace{\bigl[\,\overbrace{r + \gamma \max_{a'} Q(s', a')}^{\text{TD target}} - Q(s, a)\,\bigr]}_{\text{TD error }\delta}.$$

- **$\alpha \in (0, 1]$** is the step size (learning rate).
- The quantity in brackets is the **TD error** $\delta$ — how far off the current Q-estimate is from the one-step Bellman target.
- Crucially, the TD target uses $\max_{a'} Q(s', a')$, **not** the action actually taken next. That is why Q-learning is called **off-policy**: the value it learns assumes we will always act greedily from $s'$ onwards, regardless of what the exploration policy actually did.

**Convergence:** tabular Q-learning converges to $Q^*$ with probability 1 if every $(s, a)$ pair is visited infinitely often and the step size satisfies Robbins–Monro conditions ($\sum \alpha_t = \infty$, $\sum \alpha_t^2 < \infty$). The key practical takeaway: **you must keep exploring, forever, at least a little.**

### 5.3 The full algorithm (slide 13)

```
Initialise Q(s, a) arbitrarily (e.g., zeros)
for each episode do
    s ← initial state
    while s not terminal do
        a ← action chosen from s by some exploration policy
        take action a, observe reward r and next state s'
        Q(s, a) ← Q(s, a) + α [ r + γ max_{a'} Q(s', a') − Q(s, a) ]
        s ← s'
    end while
end for
```

### 5.4 Exploration vs exploitation, and ε-greedy

The algorithm leaves one question open: *how* does the agent pick $a$? Two extremes are both bad:

- **Greedy:** always pick $\arg\max_a Q(s, a)$. Fast but gets stuck — if the initial $Q$ accidentally assigns a higher value to the wrong action, the agent will keep picking it and never discover anything better.
- **Random:** pick uniformly. Explores everywhere but never exploits what it learned — reward stays low forever.

**ε-greedy** interpolates between the two:

$$a = \begin{cases} \text{uniform random action} & \text{with probability } \varepsilon \\ \arg\max_a Q(s, a) & \text{otherwise} \end{cases}$$

Typical practice: start with $\varepsilon = 1.0$ (pure exploration) and **decay** geometrically (e.g., multiply by $0.995$ each episode) down to a small floor $\varepsilon_{\min} \approx 0.01$. The decay schedule is a hyperparameter — too fast, and the agent commits to a suboptimal policy; too slow, and it wastes samples.

### 5.5 Grid world reference

The in-class tutorial (Section 2) implemented tabular Q-learning on a $5 \times 5$ grid world with walls. That code is the canonical reference and we will not duplicate it here. The takeaways are:

- With ~500 episodes the Q-table converges for a 25-state problem.
- The learned $V(s) = \max_a Q(s, a)$ forms a "potential field" that points towards the goal.
- The policy arrow plot shows that every non-terminal cell has learned to move along the shortest path.

### 5.6 SARSA — the on-policy cousin

A single-character change to the update rule,
$$Q(s, a) \leftarrow Q(s, a) + \alpha \bigl[r + \gamma\, Q(s', a')\; - Q(s, a)\bigr],$$
where $a'$ is the action **actually taken** next under the ε-greedy policy, gives us **SARSA** — an on-policy algorithm that learns the value of *the policy it is following*, exploration noise included. In a cliff-walking environment SARSA learns a safer (longer but less risky) path than Q-learning, because it accounts for the fact that ε-greedy occasionally steps off the cliff. This is a classical illustration of the off-policy / on-policy distinction.


## Chapter 6 — Bridging to Deep RL

*(Lecture slide 15)*

Tabular Q-learning is beautiful and provably correct — and essentially useless for any interesting problem. The reasons are two:

1. **The curse of dimensionality.** For raw Atari frames the state is an $84 \times 84 \times 4$ stack of pixels, giving $256^{84 \cdot 84 \cdot 4}$ possible states. A Q-table with that many rows cannot be written down, let alone updated. For continuous states (robot joint angles) the number of states is literally infinite.

2. **No generalisation.** A Q-table treats every state as completely independent. Even if the agent has visited a billion states that are *nearly identical* to the current one, the table tells us nothing about the current state — we have to visit it too.

Deep learning solves both problems with one idea: **function approximation**. Replace the table $Q(s, a)$ with a parameterised function $Q_\theta(s, a)$ — specifically, a neural network that *reads* $s$ and outputs $|A|$ Q-values. Similar states produce similar features in the network, and gradient descent on a shared parameter vector means an update for one state immediately generalises to all states in the same region of the representation.

Slide 15 puts it crisply:

> **Deep Learning learns the *representation*; RL defines the *objective*.**

The marriage is what unlocked the breakthroughs from Chapter 1. Chapter 7 makes this concrete with DQN.


## Chapter 7 — Deep Q-Networks (DQN)

*(Lecture slides 16–22)*

### 7.1 From a table to a network

**Q-network.** A neural network $Q_\theta$ that takes state $s$ as input and outputs a vector of $|A|$ Q-values, one per discrete action:

$$Q_\theta: s \;\longmapsto\; \bigl[Q_\theta(s, a_1),\; Q_\theta(s, a_2),\; \ldots,\; Q_\theta(s, a_{|A|})\bigr] \in \mathbb{R}^{|A|}.$$

Why output *all* actions at once and not take $a$ as input? Because Q-learning needs $\max_{a'} Q_\theta(s', a')$ at every update — computing this for $|A|$ actions is a single forward pass when the network outputs all of them.

**DQN from raw pixels (Mnih et al., 2015).** For Atari:

- Input: four most recent $84 \times 84$ grayscale frames stacked along the channel axis → a $(4, 84, 84)$ tensor. The 4-frame stack is what makes the state Markovian (one frame doesn't show velocity).
- Architecture: three convolutional layers followed by two fully-connected layers. Output dim = number of legal joystick actions (≤ 18 for Atari).
- Trained end-to-end from pixels → Q-values, with **no hand-crafted features**.

This single architecture, trained with the same hyperparameters, reached human-level play on 49 Atari games. It was the first broadly successful marriage of deep learning and RL.

### 7.2 The DQN loss — and why naive training diverges

Plugging a neural network into the Q-learning update gives the loss:

$$\mathcal{L}_{\text{naive}}(\theta) = \mathbb{E}_{(s, a, r, s')}\!\left[\bigl(r + \gamma \max_{a'} Q_\theta(s', a') - Q_\theta(s, a)\bigr)^2\right].$$

Training on this objective diverges almost immediately. Slide 20 identifies **two root causes** — they are worth thinking about carefully, because they are deep issues that come back in RLHF.

**Cause 1 — Correlated samples.** Consecutive transitions from a single trajectory are heavily correlated: in Atari, the screen barely changes between two frames, and the same action is often taken for dozens of steps. Stochastic gradient descent assumes **IID** samples; feeding it sequentially correlated data biases the gradient and can cause catastrophic forgetting. Concretely, a rollout in one part of state space nudges the weights, which changes behaviour everywhere, which changes the next rollout, which nudges the weights again — a positive feedback loop.

**Cause 2 — Non-stationary targets.** The target $r + \gamma \max_{a'} Q_\theta(s', a')$ is itself computed from $Q_\theta$, which is the thing we are trying to fit. So as soon as we take a gradient step, the target moves. This is qualitatively different from supervised learning, where the label for a given $x$ is fixed. Think of it as trying to hit a moving target that is attached to your own bow — every time you aim, the target jumps.

### 7.3 The two DQN fixes

**Experience replay** (slide 21) breaks correlation.

Keep a replay buffer $\mathcal{D}$ of recent transitions $(s, a, r, s', d)$. At each training step, instead of using the transition you just observed, sample a **random mini-batch** from $\mathcal{D}$:

$$\mathcal{L}_{\text{DQN}}(\theta) = \mathbb{E}_{(s, a, r, s') \sim \mathcal{D}}\!\left[\bigl(\text{target} - Q_\theta(s, a)\bigr)^2\right].$$

Because transitions in the buffer come from many different episodes and many different historical policies, a random mini-batch is approximately IID — exactly what SGD wants. Buffer sizes of $10^5$–$10^6$ transitions are typical.

**Target network** (slide 20) freezes the target.

Keep a separate copy of the network parameters $\theta^-$ that is used *only* for computing the target:

$$\mathcal{L}_{\text{DQN}}(\theta) = \mathbb{E}_{(s, a, r, s') \sim \mathcal{D}}\!\left[\bigl(r + \gamma \max_{a'} Q_{\theta^{-}}(s', a') - Q_\theta(s, a)\bigr)^2\right].$$

The target parameters $\theta^-$ are **copied from $\theta$ every $C$ steps** (e.g., $C = 10000$) and held fixed in between. During the $C$ steps between copies, the target is a *fixed function*, which restores the stationary-target property that supervised learning relies on.

Without **both** fixes training blows up; that is not a mild instability, it is a catastrophic divergence. This is why the 2015 DQN paper was such a big deal — it was the first time anyone made function approximation + bootstrapping + off-policy learning (Sutton's "deadly triad") play together reliably.

### 7.4 Full DQN algorithm

```
Initialise replay buffer D, Q-network parameters θ, target parameters θ⁻ ← θ
for episode = 1, 2, … do
    s ← initial state
    while not done do
        With probability ε pick a random action, else a ← argmax_a Q_θ(s, a)
        Execute a, observe r and s'
        Store (s, a, r, s', done) in D
        Sample minibatch {(s_j, a_j, r_j, s'_j, done_j)} ~ D
        y_j ← r_j + γ (1 - done_j) max_{a'} Q_{θ⁻}(s'_j, a')
        Take a gradient step on (y_j - Q_θ(s_j, a_j))² w.r.t. θ
        Every C steps, copy θ → θ⁻
        s ← s'
    end while
    Decay ε
end for
```

### 7.5 A minimal DQN update step in code


In [ ]:
# ── DQN update step (annotated, for pedagogy — not for training on Atari) ──

class QNet(nn.Module):
    def __init__(self, state_dim, n_actions, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, n_actions))
    def forward(self, x):
        return self.net(x)


def dqn_update(q_net, target_net, optimiser, batch, gamma=0.99):
    s, a, r, s_next, done = batch                       # shapes: (B, d), (B,), (B,), (B, d), (B,)

    # Current Q(s, a) from the online network — select the taken action's Q-value.
    q_sa = q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)

    # Bootstrapped target from the FROZEN target network — gradient does NOT flow here.
    with torch.no_grad():
        q_next_max = target_net(s_next).max(dim=1).values
        target = r + gamma * q_next_max * (1.0 - done)

    loss = F.mse_loss(q_sa, target)
    optimiser.zero_grad(); loss.backward(); optimiser.step()
    return loss.item()


# Smoke test: one synthetic mini-batch
torch.manual_seed(0)
q_net = QNet(state_dim=4, n_actions=2)
target_net = QNet(state_dim=4, n_actions=2)
target_net.load_state_dict(q_net.state_dict())
opt = torch.optim.Adam(q_net.parameters(), lr=1e-3)

B = 32
batch = (torch.randn(B, 4),
         torch.randint(0, 2, (B,)),
         torch.randn(B),
         torch.randn(B, 4),
         torch.zeros(B))

print('loss before update:', dqn_update(q_net, target_net, opt, batch))
print('loss after  update:', dqn_update(q_net, target_net, opt, batch))


### 7.6 Where DQN went next

DQN was not the end of the story. Three widely-used extensions:

- **Double DQN** (van Hasselt et al., 2016) — uses the online network to *select* the argmax action and the target network to *evaluate* it, cutting the positive bias of the plain $\max$ operator.
- **Dueling DQN** (Wang et al., 2016) — decomposes $Q(s, a) = V(s) + A(s, a)$ architecturally, giving better generalisation across actions.
- **Rainbow** (Hessel et al., 2018) — combines Double + Dueling + Prioritised replay + Noisy nets + Distributional RL + Multi-step returns into one agent. State-of-the-art on Atari for several years.

For this course we treat vanilla DQN as sufficient, because (a) it already contains the two conceptual fixes — replay and target network — that are the heart of deep value-based RL, and (b) the next part of the notebook moves in a very different direction: we leave pixel inputs behind and port the RL framework to *language*.


---

# Part II — Reinforcement Learning in the Age of LLMs

Chapters 8–17 retarget every concept from Part I at language models. The state becomes a prompt plus the tokens generated so far; the action becomes the next token; the reward comes from human preferences (or a verifier). Four alignment algorithms — RLHF/PPO, DPO, SLiC-HF, GRPO — progressively simplify the pipeline. These chapters cover lecture slides 23–46.


## Chapter 8 — From Games to Language

*(Lecture slides 23–25)*

### 8.1 The analogy table

Every RL concept from Part I has a direct translation to language generation:

| RL concept | Atari example | LLM example |
|---|---|---|
| Environment | Game emulator | Text completion interface |
| State $s$ | Stack of 4 frames | Prompt $x$ + tokens generated so far $y_{<t}$ |
| Action $a$ | Joystick movement | Next token $y_t$ from the vocabulary |
| Policy $\pi_\theta(a\!\mid\! s)$ | CNN over pixels | LM conditional $p_\theta(y_t \mid x, y_{<t})$ |
| Episode | One game | One full response $y = (y_1, \ldots, y_T)$ |
| Reward | Game score | Scalar score on the *whole* response |

The last row is the interesting one. In Atari the agent gets a reward at almost every frame (points ticking up). In language alignment the reward typically arrives **only once**, at the end of the response, from either:

- a **reward model** that scores $(x, y)$ as a whole (RLHF, Chapter 11);
- a **verifier** that checks whether the final answer is correct (GRPO for math/code, Chapter 16);
- a **preference pair** that ranks two full responses against each other (DPO, SLiC-HF, Chapters 14–15).

### 8.2 Credit assignment is the hard part

If the reward only arrives at the end of a 500-token response, how does the agent know *which token* deserves credit or blame? The RL framework says "update all of them proportionally to the advantage of the whole trajectory" — but doing that well is what makes LLM alignment hard. The four algorithms in the rest of Part II are, in a real sense, different answers to this one question.

### 8.3 Why open-ended language has no hand-designed reward

The dream of RL is to specify a reward function and let the agent figure everything else out. For board games this works: reward = +1 on win, −1 on loss, 0 otherwise. For Atari: reward = points.

For language, no scalar function captures what we want:

- "Helpful" — helpful *to whom*? For what *purpose*?
- "Honest" — a long hedging answer can be technically honest but unhelpfully evasive.
- "Harmless" — refusing every question is certainly harmless, and certainly useless.

The 2022 solution (InstructGPT) was to give up on hand-designing the reward and **learn it from pairwise human preferences**: given $(x, y_w, y_l)$ where a human annotator judged $y_w$ better than $y_l$, train a reward model to score $y_w$ higher than $y_l$. This is the subject of Chapter 11.


## Chapter 9 — Roadmap of Four Alignment Algorithms

*(Lecture slide 26)*

| Method | Year | Key idea | Reward model? | RL loop? | Critic? |
|---|---|---|---|---|---|
| **RLHF (PPO)** | 2022 | Full RL pipeline | ✓ | ✓ | ✓ |
| **DPO** | 2023 | Closed-form, no RL | ✗ | ✗ | ✗ |
| **SLiC-HF** | 2023 | Ranking loss | ✗ | ✗ | ✗ |
| **GRPO** | 2024 | PPO without critic | ✓ / rule | ✓ | ✗ |

The progression is one of **progressive simplification**. RLHF is the most theoretically general and the most expensive to run. Every subsequent method removes a component:

- **DPO** removes the reward model *and* the RL loop by finding the closed-form optimum of the RLHF objective.
- **SLiC-HF** removes the reference model too, reducing to a single max-margin loss.
- **GRPO** keeps the RL loop but removes the critic network — the biggest memory consumer in PPO.

Each method trades off some generality for much lower cost or much better stability. The rest of Part II works through them one at a time, starting with RLHF's three stages.


## Chapter 10 — Stage 1: Supervised Fine-Tuning (SFT)

*(Lecture slide 28)*

### 10.1 What SFT does

Start from a pretrained base LM (e.g., GPT-3, LLaMA). Collect a dataset of **demonstrations** — human-written $(x, y)$ pairs where $x$ is a prompt and $y$ is a high-quality response. Fine-tune with the standard autoregressive cross-entropy loss:

$$\mathcal{L}_{\text{SFT}}(\theta) = -\mathbb{E}_{(x, y) \sim \mathcal{D}_{\text{SFT}}}\!\left[\sum_{t=1}^{|y|} \log \pi_\theta(y_t \mid x, y_{<t})\right].$$

This is nothing more exotic than language-model training on a small, curated dataset. No RL, no reward model — just supervised learning.

### 10.2 Why SFT matters beyond being a "first stage"

SFT does two jobs in the RLHF pipeline, and it is essential to understand **both**:

**Job 1 — Initialisation for the policy.** The policy $\pi_\theta$ optimised by PPO starts from the SFT checkpoint, not from the raw pretrained model. Without this warm-start, PPO would be trying to explore in a vast prompt space with almost no signal, and would almost certainly collapse into gibberish.

**Job 2 — Reference policy for the KL penalty.** A *frozen copy* of the SFT model, denoted $\pi_{\text{ref}}$, is kept in memory throughout RLHF. Its only role is to appear in the KL penalty $\beta\, D_{\text{KL}}[\pi_\theta \,\|\, \pi_{\text{ref}}]$ (Chapter 12), anchoring $\pi_\theta$ to "reasonable" outputs and preventing **reward hacking** — degenerate outputs that maximise the reward model while being unreadable to humans.

### 10.3 What SFT does not give you

SFT teaches the model *the shape of a good answer* — format, length, tone. It does **not** teach the model the fine-grained preferences that distinguish two plausible answers. Two responses to the same prompt might be equally well-formed yet one is clearly preferable (more factual, less biased, friendlier). SFT has no way to express "this one, not that one"; that is what Stages 2 and 3 are for.

Said differently: SFT maximises $\log \pi_\theta(y \mid x)$ on a dataset of good examples. Everything it can do is inherited from the narrow distribution of human demonstrations. RLHF extends this by also using **comparisons**, which are cheaper to collect and much richer in signal.


In [ ]:
# ── A minimal SFT loss (conceptual) ─────────────────────────────────────────

def sft_loss(logits, target_ids, ignore_index=-100):
    """Standard autoregressive CE loss.
    logits:     (B, T, V)    — model outputs
    target_ids: (B, T)       — shifted input ids, -100 for prompt tokens
    Returns scalar mean loss.
    """
    B, T, V = logits.shape
    return F.cross_entropy(logits.reshape(B * T, V),
                           target_ids.reshape(B * T),
                           ignore_index=ignore_index)


# Tiny smoke test
B, T, V = 2, 5, 10
logits = torch.randn(B, T, V)
target = torch.tensor([[-100, -100, 3, 7, 1],
                       [-100,    4, 2, 5, 0]])    # -100 masks prompt tokens
print('SFT loss:', sft_loss(logits, target).item())


## Chapter 11 — Stage 2: The Reward Model (Bradley–Terry)

*(Lecture slide 29)*

### 11.1 The data

Reward-model training uses **comparison data**: triples $(x, y_w, y_l)$ where $x$ is a prompt, $y_w$ is the response a human preferred, and $y_l$ is the response they rejected. InstructGPT collected roughly 33K such comparisons (small by NLP standards) and that was enough to move a 1.3B model past GPT-3 175B on human evaluations.

Why comparisons and not absolute ratings? Because humans are *much* more consistent at pairwise judgments than at assigning a numerical score to a single response in isolation.

### 11.2 The Bradley–Terry preference model

We need a probabilistic model that turns a scalar score $r(x, y)$ into a preference probability. The standard choice is the **Bradley–Terry model** (Bradley & Terry, 1952), which postulates that each item has a latent "worth" $w_i = e^{r_i}$ and that

$$P(i \text{ preferred over } j) = \frac{w_i}{w_i + w_j} = \frac{e^{r_i}}{e^{r_i} + e^{r_j}} = \sigma(r_i - r_j),$$

where $\sigma$ is the logistic sigmoid. Three properties make this the natural choice:

1. **Only differences matter.** Multiplying both worths by the same constant does not change the probability — so $r$ is only identified up to an additive constant. This exactly matches the intuition that "better than" is a *relative* judgment.
2. **Transitivity.** If item $i$ has high probability of beating $j$, and $j$ of beating $k$, then $i$ beats $k$. This matches how humans usually reason about preferences.
3. **Equivalence to logistic regression.** Fitting BT is *identical* to logistic regression on score differences, so every standard trick for logistic regression carries over.

### 11.3 The reward-model loss

Maximise the likelihood of the observed preferences under the BT model:

$$\mathcal{L}_{\text{RM}}(\phi) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}_{\text{pref}}}\!\left[\log \sigma\bigl(r_\phi(x, y_w) - r_\phi(x, y_l)\bigr)\right].$$

This is the same binary cross-entropy loss you would use for any logistic regression.

### 11.4 Architecture

The reward model is *not* a fresh network. It is the SFT model with the last layer replaced by a linear projection to a single scalar — exactly the classification-head trick from BERT fine-tuning. Concretely, you take the hidden state at the final token position and project it to $\mathbb{R}$.

This architectural choice has a practical consequence: the reward model is the same size as the SFT model. If the policy is a 7B LLM, the reward model is also 7B parameters. We will care about this in Chapter 12 when we count memory.


In [ ]:
# ── Bradley–Terry reward-model loss ────────────────────────────────────────

def rm_loss(r_w, r_l):
    """r_w, r_l: (B,) scalar scores from the reward model for chosen / rejected responses."""
    return -F.logsigmoid(r_w - r_l).mean()


# Synthetic example: the reward model is initially uncalibrated
torch.manual_seed(0)
B = 8
r_w = torch.randn(B, requires_grad=True)   # scores for preferred responses
r_l = torch.randn(B, requires_grad=True)   # scores for rejected responses

loss = rm_loss(r_w, r_l)
loss.backward()

print(f'Initial RM loss: {loss.item():.4f}')
print(f'Gradient on r_w (preferred):  {r_w.grad.tolist()}')
print(f'Gradient on r_l (rejected):   {r_l.grad.tolist()}')
print()
print('Note: grad on r_w is always negative, so SGD pushes r_w UP.')
print('      grad on r_l is always positive, so SGD pushes r_l DOWN.')
print('      Magnitude is σ(-(r_w - r_l)) — large when the model gets it wrong.')


Take a second look at the gradients above. The weight assigned to each preference pair is $\sigma(-(r_w - r_l))$ — which is **large when $r_w < r_l$** (i.e., the reward model currently thinks the wrong answer is better) and **small when $r_w \gg r_l$** (the model already gets it right). This is *implicit hard-example mining*: the training signal automatically concentrates on the pairs the model finds difficult.

The same structural feature reappears in the DPO gradient in Chapter 14, for exactly the same mathematical reason.


## Chapter 12 — Stage 3: RLHF Fine-Tuning with PPO

*(Lecture slides 30–32)*

### 12.1 The RLHF objective

Once we have a reward model $r_\phi$ and a reference policy $\pi_{\text{ref}}$, Stage 3 trains the policy $\pi_\theta$ to maximise reward while staying close to the reference:

$$\boxed{\;\max_\theta\; \mathbb{E}_{x \sim \mathcal{D},\, y \sim \pi_\theta(\cdot \mid x)}\!\left[r_\phi(x, y)\right]\;-\;\beta\, \mathbb{E}_{x}\!\left[D_{\text{KL}}\bigl[\pi_\theta(\cdot \mid x)\,\|\,\pi_{\text{ref}}(\cdot \mid x)\bigr]\right]\;}$$

The scalar $\beta$ controls the trade-off between chasing reward and staying close to $\pi_{\text{ref}}$.

**Why the KL penalty?**

Without the KL term, the policy would exploit *every* imperfection of the reward model — outputting exotic high-scoring gibberish that the reward model happens to like but humans don't. This is called **reward hacking** and is a real failure mode. The KL penalty constrains the policy to a neighbourhood of the SFT distribution where the reward model was trained and is still trustworthy.

### 12.2 Per-token reward decomposition

An LLM generates tokens one at a time, but the reward arrives at the end. How do we turn a whole-response reward into per-token targets for RL?

The standard decomposition (used in InstructGPT and essentially every open-source RLHF implementation):

- **For intermediate tokens $t < T$:**
  $$\tilde r_t = -\beta \bigl[\log \pi_\theta(y_t \mid x, y_{<t}) - \log \pi_{\text{ref}}(y_t \mid x, y_{<t})\bigr]$$
  — a *per-token KL penalty only*, no contribution from the reward model.

- **For the final token $t = T$:**
  $$\tilde r_T = r_\phi(x, y) \;-\; \beta \bigl[\log \pi_\theta(y_T \mid \ldots) - \log \pi_{\text{ref}}(y_T \mid \ldots)\bigr]$$
  — the reward model's whole-response score, plus the final-token KL.

This pushes the *reward model's signal* entirely to the last step, while the KL penalty is paid every step. The advantage estimator (GAE, below) then propagates the terminal reward backwards through time, allocating credit across tokens.

### 12.3 PPO's clipped surrogate objective

PPO (Schulman et al., 2017) is a policy-gradient algorithm. Its loss uses the **importance sampling ratio** between the current policy and the "old" policy that produced the rollout:

$$\rho_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}.$$

If we simply maximised $\mathbb{E}[\rho_t \hat A_t]$ where $\hat A_t$ is the advantage estimate, the objective would be correct in expectation but catastrophically unstable: a single large $\rho_t \hat A_t$ can push the policy into a region where the approximation is invalid and the collected data is useless.

PPO's **clipped surrogate** fixes this by capping how much $\rho_t$ can move the objective:

$$\boxed{\;\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t\!\left[\min\bigl(\rho_t(\theta)\, \hat A_t,\; \text{clip}(\rho_t(\theta), 1 - \varepsilon, 1 + \varepsilon)\, \hat A_t\bigr)\right]\;}$$

Two cases to internalise:

- **$\hat A_t > 0$** (good action). The unclipped term wants to keep increasing $\rho_t$ — make the action more and more likely. Clipping at $1 + \varepsilon$ caps the bonus: beyond that point the gradient is zero and PPO stops making "good" actions even more likely.
- **$\hat A_t < 0$** (bad action). The unclipped term wants to keep decreasing $\rho_t$. Clipping at $1 - \varepsilon$ caps the penalty: beyond that point the gradient is zero and PPO stops making "bad" actions even more unlikely.

Either way, the clip enforces a **trust region** — the policy is never allowed to change by more than about $\varepsilon$ per update step. Typical $\varepsilon = 0.2$.

### 12.4 Four models in memory

In standard RLHF, four separate models need to sit on GPUs simultaneously:

| Model | Role | Trainable? | Size |
|---|---|---|---|
| **Policy** $\pi_\theta$ | The LLM being optimised | Yes | $N$ params |
| **Reference** $\pi_{\text{ref}}$ | Frozen SFT copy — anchor for KL penalty | No | $N$ params |
| **Reward** $r_\phi$ | Scores whole responses | No | $\approx N$ params |
| **Value** $V_\psi$ | Critic for advantage estimation | Yes | $\approx N$ params |

So a 7B policy implies **~28B total parameters** to hold in memory. In fp16 that is about 56 GB just for weights. Add Adam optimizer states for the two trainable models (momentum + variance = 2× the parameters each, often in fp32 = 4× the parameters per trainable model), and you are looking at **100+ GB of GPU memory** to RLHF a 7B model — on top of activations and KV caches for generation.

This is precisely the engineering pain that DPO, SLiC-HF, and GRPO are trying to relieve.


In [ ]:
# ── PPO clipping visualised (same as the tutorial figure, for reference) ───

ratio = torch.linspace(0.0, 2.5, 500)
eps = 0.2
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for ax, A, title in [(ax1, +1.0, 'Â > 0 (good action)'),
                     (ax2, -1.0, 'Â < 0 (bad action)')]:
    unclipped = ratio * A
    clipped = torch.clamp(ratio, 1 - eps, 1 + eps) * A
    ppo = torch.min(unclipped, clipped)
    ax.plot(ratio, unclipped, '--', label='unclipped')
    ax.plot(ratio, clipped, '--', label='clipped only')
    ax.plot(ratio, ppo, linewidth=2.2, label='PPO min(·,·)')
    ax.axvline(1 - eps, color='r', ls=':', alpha=0.4)
    ax.axvline(1 + eps, color='r', ls=':', alpha=0.4)
    ax.set_xlabel('ratio ρ(θ)'); ax.set_ylabel('objective')
    ax.set_title(title); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


## Chapter 13 — InstructGPT: The Empirical Payoff

*(Lecture slide 33)*

InstructGPT (Ouyang et al., 2022) is the paper that established the now-standard three-stage recipe (SFT → Reward Model → PPO) and produced the model family that ChatGPT was built on. Three results are worth remembering.

**1. A 1.3B InstructGPT was preferred over a 175B GPT-3.** That is a 100× reduction in parameter count — bought entirely by alignment, with no improvement in raw capability. Alignment matters at least as much as scale.

**2. The datasets are surprisingly small.**

- SFT demonstrations: ~13K examples
- Preference comparisons: ~33K pairs
- PPO prompts: ~31K prompts

Compared to the trillions of tokens used for pretraining, this is a rounding error. The lesson is that the hard part of alignment is not data volume but data *quality* — careful labelling instructions and experienced annotators.

**3. The PPO-ptx variant.** Pure RLHF fine-tuning causes the model to forget some of its pretraining knowledge — reward optimisation narrows the output distribution. InstructGPT added a small *pretraining loss term* (the "ptx" in PPO-ptx) that keeps the model honest on the original LM objective while PPO is running. This is a common trick in production RLHF systems.

**Remaining challenges** (all still active research in 2025):

- **Hallucinations** are reduced but not solved — RLHF can shape what the model says, but it cannot fix things the base model never knew.
- **Reward hacking** is always lurking. The reward model is a learned, imperfect proxy for human judgment; PPO is exceptionally good at finding its blind spots.
- **Training is unstable and expensive.** The four-model architecture (Chapter 12) is hard to tune and easy to blow up. This is the direct motivation for DPO, SLiC-HF, and GRPO.


## Chapter 14 — DPO: Direct Preference Optimization

*(Lecture slides 34–38)*

### 14.1 The guiding question

Rafailov et al. (2023) asked: "The RLHF objective is a constrained optimisation problem. Does it have a *closed-form* solution? And if so, can we skip the RL loop entirely?" The answer turned out to be yes — and the resulting method, DPO, has become the default alignment algorithm for open-source models (LLaMA 2/3, Mistral, Qwen, etc.) where simplicity matters more than absolute peak performance.

### 14.2 The derivation in full

**Step 1 — Closed-form optimal policy.** Fix a prompt $x$ and write $\pi$ for $\pi(\cdot \mid x)$ and $\pi_{\text{ref}}$ for $\pi_{\text{ref}}(\cdot \mid x)$. The per-prompt RLHF objective is

$$\max_\pi \sum_y \pi(y) r(x, y) - \beta \sum_y \pi(y) \log \frac{\pi(y)}{\pi_{\text{ref}}(y)}.$$

This is concave in $\pi$, and constrained by $\sum_y \pi(y) = 1$, $\pi(y) \geq 0$. Form the Lagrangian and differentiate with respect to $\pi(y)$:

$$r(x, y) - \beta\bigl(1 + \log \pi(y) - \log \pi_{\text{ref}}(y)\bigr) - \lambda = 0,$$

solve for $\log \pi(y)$:

$$\log \pi^*(y) = \log \pi_{\text{ref}}(y) + \frac{r(x, y)}{\beta} - \bigl(1 + \tfrac{\lambda}{\beta}\bigr).$$

Exponentiate and normalise so the distribution sums to 1:

$$\boxed{\;\pi^*(y \mid x) = \frac{1}{Z(x)}\, \pi_{\text{ref}}(y \mid x)\, \exp\!\left(\frac{r(x, y)}{\beta}\right),\qquad Z(x) = \sum_y \pi_{\text{ref}}(y \mid x)\, \exp\!\left(\frac{r(x, y)}{\beta}\right)\;}$$

This is the **exponentially-tilted reference distribution** — intuitively, you reweight $\pi_{\text{ref}}$ by a factor that is large where the reward is high.

**Step 2 — Reparameterise the reward.** Solve the boxed equation above for $r(x, y)$:

$$r(x, y) = \beta \log \frac{\pi^*(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x).$$

The second term $\beta \log Z(x)$ depends only on $x$, not on $y$. Keep this in mind — it is about to cancel.

**Step 3 — Substitute into Bradley–Terry.** The reward-model loss from Chapter 11 was $-\log \sigma(r(x, y_w) - r(x, y_l))$. Plug the reparameterised expression for $r$ into the *difference*:

$$
\begin{aligned}
r(x, y_w) - r(x, y_l) &= \beta \log \frac{\pi^*(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} + \beta \log Z(x) - \beta \log \frac{\pi^*(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} - \beta \log Z(x)\\
&= \beta\!\left[\log \frac{\pi^*(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \log \frac{\pi^*(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\right].
\end{aligned}
$$

**The $Z(x)$ terms cancel** — because the Bradley–Terry model depends only on score *differences*, and $Z(x)$ is the same for both responses to the same prompt. This is the magic step.

**Step 4 — The DPO loss.** Replace $\pi^*$ (which we don't have) with the learnable $\pi_\theta$, and maximise the likelihood of the observed preferences:

$$\boxed{\;\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x, y_w, y_l)}\!\left[\log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\right)\right]\;}$$

It is a plain binary cross-entropy loss on preference pairs, evaluated with log-probabilities from two models. No reward model, no PPO loop, no critic, no online generation — just offline supervised-style training on the preference dataset.

**Theoretical guarantee.** The derivation is *exact* under the assumption that the reward is expressible in the reparameterised form. In particular, DPO provably optimises the same KL-constrained objective as RLHF — without ever explicitly representing a reward function. This is a striking result.

### 14.3 The DPO gradient is implicit hard-example mining

Write $h_\theta(x, y_w, y_l) = \beta \log \frac{\pi_\theta(y_w)}{\pi_{\text{ref}}(y_w)} - \beta \log \frac{\pi_\theta(y_l)}{\pi_{\text{ref}}(y_l)}$. Then $\mathcal{L}_{\text{DPO}} = -\log \sigma(h_\theta)$. Its gradient is:

$$\nabla_\theta \mathcal{L}_{\text{DPO}} = -\underbrace{\sigma(-h_\theta)}_{\text{per-sample weight}} \cdot \beta \cdot \bigl[\nabla_\theta \log \pi_\theta(y_w \mid x) - \nabla_\theta \log \pi_\theta(y_l \mid x)\bigr].$$

Two things to notice:

1. The bracket **pushes up** $\log \pi_\theta(y_w)$ and **pushes down** $\log \pi_\theta(y_l)$ — exactly what we want for preference learning.
2. The scalar weight $\sigma(-h_\theta)$ is **large when $h_\theta < 0$** — i.e., when the model currently thinks $y_l$ is better than $y_w$ (it is wrong on this pair). It is **small when $h_\theta \gg 0$** (the model already gets the pair right).

So DPO automatically focuses learning on the pairs the model is currently wrong about — *implicit hard-example mining*, for free. The same structure appeared in the reward-model loss in Chapter 11; it is the same sigmoid showing up for the same reason.

### 14.4 The role of $\beta$

- **$\beta \to 0$**: no KL anchor. The policy can drift arbitrarily far from $\pi_{\text{ref}}$, overfitting to the preference dataset.
- **$\beta \to \infty$**: the KL penalty dominates. The policy is pinned to $\pi_{\text{ref}}$ and barely learns anything.
- Typical values in practice are $\beta \in [0.01, 0.5]$ with $0.1$ being a common default.

### 14.5 Known limitations

DPO is beautiful but not a free lunch:

- **Offline distribution mismatch.** DPO only sees pairs from a fixed dataset. Unlike PPO, it never gets to generate new samples from its current policy. If the optimal policy's distribution differs a lot from where the dataset was collected, DPO can under-perform.
- **Sensitive to $\beta$.** The best $\beta$ depends on the size and noise of the dataset, and there is no principled way to pick it short of a sweep.
- **No explicit reward.** You cannot reuse a trained DPO model to rank new responses the way you can with a reward model. The reward is implicit in $\log(\pi_\theta / \pi_{\text{ref}})$, which is harder to query.
- **Label noise sensitivity.** A small fraction of mis-labelled preference pairs can have outsized effect, because the gradient weight $\sigma(-h_\theta)$ is largest precisely when the model disagrees with the (possibly wrong) label.

### 14.6 DPO gradient in code


In [ ]:
# ── DPO loss + inspect the implicit hard-example weight ───────────────────

def dpo_loss_and_weight(logp_w, logp_l, ref_w, ref_l, beta=0.1):
    h = beta * ((logp_w - ref_w) - (logp_l - ref_l))
    loss = -F.logsigmoid(h).mean()
    per_sample_weight = torch.sigmoid(-h)   # this is the hard-example mining weight
    return loss, h, per_sample_weight


# Construct three pairs with very different difficulty:
#   pair 0 — model is SURE y_w is preferred (already correct, easy)
#   pair 1 — model is 50/50
#   pair 2 — model is SURE y_l is preferred (wrong, hard)
logp_w = torch.tensor([ -1.0, -1.0, -4.0])
logp_l = torch.tensor([ -4.0, -1.0, -1.0])
ref_w  = torch.tensor([ -2.0, -2.0, -2.0])
ref_l  = torch.tensor([ -2.0, -2.0, -2.0])

loss, h, w = dpo_loss_and_weight(logp_w, logp_l, ref_w, ref_l, beta=1.0)
for i in range(3):
    label = ['easy (already correct)', 'borderline', 'hard (currently wrong)'][i]
    print(f'pair {i} [{label:25}]  h = {h[i]:+.2f}   per-sample weight = {w[i]:.3f}')
print()
print(f'mean DPO loss across all 3 pairs: {loss.item():.4f}')


The per-sample weights tell the whole story: the "easy" pair contributes almost nothing to the gradient, the borderline pair contributes $\approx 0.5$, and the hard pair contributes $\approx 1.0$. DPO is *automatically* training on the pairs the model cannot yet rank — you don't need a curriculum, the loss function builds one for you.


## Chapter 15 — SLiC-HF: Sequence Likelihood Calibration

*(Lecture slide 39)*

### 15.1 Even simpler than DPO

Zhao et al. (2023) asked: *can we drop the reference model too?* Their answer is SLiC-HF, which replaces the log-sigmoid of DPO with a plain max-margin ranking loss directly on the policy's log-likelihoods:

$$\boxed{\;\mathcal{L}_{\text{SLiC}}(\theta) = \mathbb{E}_{(x, y_w, y_l)}\!\left[\max\!\Bigl(0,\; \delta - \log \pi_\theta(y_w \mid x) + \log \pi_\theta(y_l \mid x)\Bigr)\right]\;}$$

where $\delta > 0$ is a hyperparameter margin. The intuition:

- The preferred response should have log-likelihood at least $\delta$ higher than the rejected one.
- Once that margin is met, the loss is **exactly zero** — no more gradient, the model has "finished" learning this pair.
- When the margin is not met, the loss increases linearly with the shortfall.

Compared to DPO, two things have changed:

1. **No reference model** — the loss acts directly on $\log \pi_\theta$, not on log-ratios $\log(\pi_\theta / \pi_{\text{ref}})$. Saves one large model in memory.
2. **Hinge loss instead of log-sigmoid** — gradients are exactly zero for easy pairs, exactly constant for hard pairs. Simpler and sharper than DPO's smooth weighting.

### 15.2 One-sided regularisation

Dropping the reference model entirely risks catastrophic forgetting — the policy might increase $\log \pi_\theta(y_w)$ while also drifting far from reasonable outputs. SLiC-HF adds an **asymmetric** regularisation term that only kicks in when the policy starts forgetting its reference behaviour:

$$\mathcal{L} = \mathcal{L}_{\text{SLiC}} + \lambda \cdot \mathcal{L}_{\text{reg}},$$

where $\mathcal{L}_{\text{reg}}$ penalises the policy only when $\log \pi_\theta$ drops *below* $\log \pi_{\text{ref}}$ on the preferred responses. It does not care if the policy puts extra probability on $y_w$ — that is what we wanted anyway. This is sometimes called a "one-sided KL" or "one-sided calibration" term.

In practice SLiC-HF uses the *SFT* model as both the initialisation and the implicit reference for the regulariser — the "reference" model is still present, but only for the regularisation, not in the main loss.

### 15.3 Cross-entropy calibration variant

The paper also proposes a soft variant using cross-entropy directly on the policy likelihoods:

$$\mathcal{L}_{\text{SLiC-CE}} = -\log \frac{\pi_\theta(y_w \mid x)}{\pi_\theta(y_w \mid x) + \pi_\theta(y_l \mid x)}.$$

This looks like DPO without the $\pi_{\text{ref}}$ terms, and it is. Both the hinge and cross-entropy forms are used in practice; the hinge form is cleaner for analysis and is what we will code.

### 15.4 DPO vs SLiC-HF at a glance

| Aspect | DPO | SLiC-HF |
|---|---|---|
| Loss type | $-\log \sigma$ (soft) | $\max(0, \delta - \cdot)$ (hinge) |
| Reference model in main loss? | Yes | No |
| Gradient on easy pairs | Small but nonzero | Exactly zero |
| Gradient on hard pairs | Bounded ($\leq 1$) | Constant $\beta$ |
| Regularisation | Implicit via $\pi_{\text{ref}}$ | Explicit, one-sided, on SFT |


In [ ]:
# ── SLiC-HF max-margin loss and its zero-gradient behaviour ─────────────────

def slic_loss(logp_w, logp_l, delta=1.0):
    return F.relu(delta - logp_w + logp_l).mean()


# Same three pairs as the DPO example
logp_w = torch.tensor([ -1.0, -1.0, -4.0], requires_grad=True)
logp_l = torch.tensor([ -4.0, -1.0, -1.0], requires_grad=True)

loss = slic_loss(logp_w, logp_l, delta=1.0)
loss.backward()
print(f'SLiC loss (mean over 3 pairs): {loss.item():.4f}')
print('per-pair margin (logp_w - logp_l):', (logp_w - logp_l).detach().tolist())
print('grad on logp_w:', logp_w.grad.tolist())
print('grad on logp_l:', logp_l.grad.tolist())
print()
print('Observe: pair 0 has margin = 3.0 >= δ = 1.0, so its contribution is ZERO — no gradient at all.')
print('pair 1 has margin = 0.0, pair 2 has margin = -3.0; both contribute constant gradient 1/3 per sample.')


## Chapter 16 — GRPO: Group Relative Policy Optimization

*(Lecture slides 40–44)*

### 16.1 What GRPO removes

Among the four models in RLHF (Chapter 12), the **value network** (critic) is the most expensive in practice: it has to be the same size as the policy, because it reads the same long token sequences and outputs a scalar at every position. For a 7B policy the critic is another 7B — and for DeepSeek-V3, whose policy is a 671B MoE, a same-size critic is literally unaffordable.

GRPO (Shao et al., "DeepSeekMath", 2024) removes the critic. The advantage $\hat A_t$ — which PPO computes with Generalised Advantage Estimation on top of $V_\psi$ — is replaced by **group-level statistics over reward samples**. No extra neural network, zero added parameters.

### 16.2 Algorithm

For each prompt $x$:

**Step 1 — Group sampling.** Sample $G$ responses $\{y_1, y_2, \ldots, y_G\}$ from the current policy $\pi_{\theta_{\text{old}}}$. Typical $G = 16$–$64$. These are all different rollouts from the *same* prompt.

**Step 2 — Reward each sample.** Compute $r_i = r_\phi(x, y_i)$ (or a rule-based verifier, e.g., "does the math answer equal the ground truth?").

**Step 3 — Group-relative advantage.** Normalise the rewards within the group:

$$\boxed{\;\hat A_i = \frac{r_i - \text{mean}(r_1, \ldots, r_G)}{\text{std}(r_1, \ldots, r_G) + \epsilon}\;}$$

Observations that will not get any better than the group average get *negative* advantage; outliers on the high end get positive advantage. This is **self-referential**: the baseline comes from the samples themselves, not from any learned value function.

**Step 4 — PPO-clipped surrogate, with the group-relative advantage.** The per-token ratio is the usual thing:

$$\rho_{i, t}(\theta) = \frac{\pi_\theta(y_{i, t} \mid x, y_{i, <t})}{\pi_{\theta_{\text{old}}}(y_{i, t} \mid x, y_{i, <t})}.$$

The GRPO objective is:

$$\mathcal{L}_{\text{GRPO}}(\theta) = -\frac{1}{G}\sum_{i=1}^{G} \frac{1}{|y_i|} \sum_{t=1}^{|y_i|} \min\!\bigl(\rho_{i,t}\, \hat A_i,\; \text{clip}(\rho_{i,t}, 1-\varepsilon, 1+\varepsilon)\, \hat A_i\bigr) + \beta\, D_{\text{KL}}\!\bigl[\pi_\theta \,\|\, \pi_{\text{ref}}\bigr].$$

Notice the outcome-level advantage $\hat A_i$ is *broadcast to all tokens in the same response*. There is no per-token value function; every token of response $i$ shares its fate with the sample's outcome. This is a much coarser credit assignment than GAE, but it is also much cheaper and — on tasks with verifiable terminal rewards — empirically works very well.

### 16.3 Why the group-relative baseline works

GRPO is, mathematically, **REINFORCE with a group-mean baseline** plus PPO clipping plus a KL penalty. Let's unpack that.

Plain REINFORCE: $\nabla_\theta \mathbb{E}[R] = \mathbb{E}[R \cdot \nabla_\theta \log \pi_\theta]$. High variance, because $R$ has a large mean and the gradient is the full reward times the score function.

REINFORCE with a baseline $b$: $\nabla_\theta \mathbb{E}[R] = \mathbb{E}[(R - b) \cdot \nabla_\theta \log \pi_\theta]$ — provided $b$ does not depend on the action, subtracting $b$ does not bias the expectation. Variance can drop dramatically if $b$ is close to $\mathbb{E}[R]$.

In PPO, the baseline is $V_\psi(s)$, learned by a critic network. In GRPO, the baseline is $\text{mean}(r_1, \ldots, r_G)$ — an unbiased Monte-Carlo estimate of the same quantity *using the samples you already generated*. It costs zero extra parameters and zero extra gradient steps; you get it by running `torch.mean` over the reward tensor.

The $\text{std}$ normalisation in the denominator is a further variance-reduction trick: it makes the advantage scale-invariant, so the effective learning rate does not depend on the reward magnitude.

### 16.4 Where the savings come from

Memory accounting for a $N$-parameter policy:

| Component | PPO | GRPO |
|---|---|---|
| Policy | $N$ | $N$ |
| Reference | $N$ | $N$ |
| Reward | $\approx N$ | $\approx N$ (or rule-based: 0) |
| **Value / critic** | $\approx N$ | **0** |
| **Total** | **$\approx 4N$** | **$\approx 3N$** (or $2N$ with rules) |

For a 7B policy: 56 GB → 42 GB (or 28 GB). For a 671B DeepSeek-V3 policy the critic would be prohibitive in *any* hardware budget; GRPO is effectively the only feasible option.

The trade-off is that GRPO has to **sample $G$ responses per prompt**, so generation cost is $G \times$ higher. For $G \approx 32$ this is non-trivial, but still cheaper than training a full critic at 7B+ scales.

### 16.5 DeepSeek-R1 — the R1-Zero result

GRPO is the reason DeepSeek-R1 (January 2025) became a landmark model. Two variants were released:

- **R1-Zero** — trained *purely* with GRPO on top of the DeepSeek-V3 base model, using rule-based rewards (is the math answer right? does the code pass the tests?). No supervised fine-tuning stage at all. The model **spontaneously developed long chains of thought**, self-verification passes, and what the paper calls "aha moments" — places where it realises a mistake mid-reasoning and starts over. All of this emerged from RL alone, with no demonstrations.

- **R1** — adds a cold-start SFT on a small curated dataset of long CoT examples, then GRPO, then rejection sampling, for a production-ready model. The paper argues the cold start mainly helps with *readability* — the R1-Zero reasoning traces work but are hard for humans to follow.

R1-Zero is the empirical proof that **on-policy RL with sparse verifiable rewards can train capable reasoning models** without supervised data. Before R1, this was debated; after R1, it is the default assumption.

### 16.6 GRPO failure mode

The Achilles heel of GRPO: **if every response in a group gets the same reward, the advantages degenerate.**

- All-correct group: $\hat A_i = 0$ for all $i$, no signal, gradient update is zero. Fine, the model already solved this prompt.
- All-wrong group: $\hat A_i = 0 / 0$ — or, with the $\epsilon$ denominator, advantages become an arbitrary small number. No useful signal, and the $\epsilon$-regularised result is noise rather than information.

In the second case the prompt is effectively wasted training compute. Large real-world runs therefore filter out prompts where the policy is "stuck" (nothing works) until it has become capable enough to sometimes succeed. This is a form of automatic curriculum.


In [ ]:
# ── GRPO step, failure-mode demo, and baseline-variance check ──────────────

def grpo_advantages(rewards):
    return (rewards - rewards.mean()) / (rewards.std() + 1e-8)


# Case 1: mixed outcomes → healthy advantages
r1 = torch.tensor([1., 0., 0., 1., 0., 0., 0., 1.])
print('Mixed group:')
print('  rewards   :', r1.tolist())
print('  advantages:', [f'{a:+.3f}' for a in grpo_advantages(r1).tolist()])

# Case 2: all-correct → zero advantage everywhere
r2 = torch.ones(8)
print()
print('All-correct group:')
print('  rewards   :', r2.tolist())
print('  advantages:', [f'{a:+.3f}' for a in grpo_advantages(r2).tolist()])
print('  → no gradient signal, fine (problem already solved).')

# Case 3: all-wrong → degenerate advantages (noise only)
r3 = torch.zeros(8)
print()
print('All-wrong group:')
print('  rewards   :', r3.tolist())
print('  advantages:', [f'{a:+.3f}' for a in grpo_advantages(r3).tolist()])
print('  → wasted compute; in practice you filter out such prompts.')

# Sanity check: variance reduction from baseline subtraction
rewards_many = torch.rand(10000) * 10.0   # raw rewards in [0, 10]
adv_many = grpo_advantages(rewards_many)
print()
print('Variance reduction check (10000 samples):')
print(f'  var(rewards)    = {rewards_many.var().item():.3f}')
print(f'  var(advantages) = {adv_many.var().item():.3f}   (should be ~1 by construction)')


## Chapter 17 — Unified Comparison of All Four Methods

*(Lecture slide 45)*

| Aspect | RLHF (PPO) | DPO | SLiC-HF | GRPO |
|---|---|---|---|---|
| Pipeline | SFT → RM → PPO | SFT → DPO | SFT → Ranking | SFT → Group PPO |
| Reward model | Trained (large) | Implicit (in log-ratio) | Not needed | Trained or rule-based |
| Critic (value net) | Yes, full size | — | — | No (group statistics) |
| Reference model | Yes | Yes | Only in regulariser | Yes |
| Models in GPU memory | **4** | **2** | **1–2** | **3** |
| Online generation during training? | Yes | No | No | **Yes** |
| Main loss type | PPO clipped surrogate | Log-ratio cross-entropy | Hinge / margin | PPO clipped + group advantage |
| Training stability | Low | High | High | Medium–High |
| Best for | General alignment, well-funded teams | General alignment, easy to reproduce | Simple preference calibration | Tasks with verifiable rewards (math, code) |
| Notable users | ChatGPT, early Claude | Llama 2/3, Mistral, open source | Research, Google T5 calibration | DeepSeek-R1, reasoning RL |

### 17.1 A decision flowchart

Consider these questions, in order:

1. **Do you have verifiable rewards?** (Math, code, unit tests, formal grammars.)
   → **GRPO** is the natural choice. It was designed for this case and delivers the biggest empirical wins.

2. **Do you have a preference dataset and want minimal engineering?**
   → **DPO**. Offline, stable, one loss, no RL loop. The default for open-source alignment in 2024–2025.

3. **Do you need the absolute best quality and have the compute budget?**
   → **RLHF / PPO**. Still the gold standard at the top of the leaderboard for proprietary models, despite the engineering cost.

4. **Do you want the absolute simplest possible loss, and are willing to trade some quality?**
   → **SLiC-HF**. One max-margin loss, no reference model in the main term, very small implementation.

### 17.2 What to take away

The six years of alignment research from InstructGPT (2022) to DeepSeek-R1 (2025) trace a clear arc: **move from complex RL pipelines toward simpler, more targeted alternatives** — while still leaving PPO-style on-policy RL for the cases where nothing else works. GRPO is the end of a loop: it started by simplifying RLHF (removing the critic), and R1-Zero showed that well-tuned on-policy RL is *still* the right tool for open-ended reasoning. The methods coexist; which one you reach for depends on what data, compute, and reward structure you have.


## Chapter 18 — References and Further Reading

### Lecture references (slide 46)

- Ouyang et al. (2022). *Training Language Models to Follow Instructions with Human Feedback.* NeurIPS. — The InstructGPT paper; canonical reference for the 3-stage RLHF recipe.
- Schulman et al. (2017). *Proximal Policy Optimization Algorithms.* arXiv:1707.06347. — The original PPO paper; read Sections 3–4 for the clipped objective intuition.
- Christiano et al. (2017). *Deep Reinforcement Learning from Human Preferences.* NeurIPS. — Pre-LLM RLHF, worth reading for the preference-learning framework.
- Stiennon et al. (2020). *Learning to Summarize with Human Feedback.* NeurIPS. — The first convincing large-scale RLHF result.
- Rafailov et al. (2023). *Direct Preference Optimization: Your Language Model is Secretly a Reward Model.* NeurIPS. — The DPO paper.
- Zhao et al. (2023). *SLiC-HF: Sequence Likelihood Calibration with Human Feedback.* arXiv:2305.10425.
- Yuan et al. (2023). *RRHF: Rank Responses to Align Language Models with Human Feedback without Tears.* arXiv:2304.05302.
- Shao et al. (2024). *DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models.* arXiv:2402.03300. — The paper that introduced GRPO.
- DeepSeek-AI (2025). *DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning.* arXiv:2501.12948. — The R1 / R1-Zero paper; read Sections 2–4.
- Bradley & Terry (1952). *Rank Analysis of Incomplete Block Designs I. The Method of Paired Comparisons.* Biometrika.

### Recommended additional reading

- **Sutton & Barto**, *Reinforcement Learning: An Introduction* (2nd edition, 2018). Chapters 3 (MDPs), 4 (Dynamic Programming), 6 (TD learning), 9 (on-policy approximation). The canonical RL textbook; free PDF at the author's site.
- **David Silver's Deep RL course** (UCL / DeepMind, 2015). Lecture slides and videos on YouTube; slides 1–6 overlap tightly with Part I here.
- **OpenAI Spinning Up in Deep RL** (Achiam, 2018). A practical introduction to policy-gradient methods with accompanying implementations; excellent if you want to see PPO and friends in code.
- **Hugging Face Alignment Handbook** (GitHub). Practical RLHF / DPO / ORPO implementations at the 7B scale; the easiest starting point if you want to run any of this yourself.
- **The Huyen RLHF post** — Chip Huyen's *"RLHF: Reinforcement Learning from Human Feedback"* article. A high-quality engineering-oriented overview; complements this notebook's mathematical take.
